# 2.5 — Minh họa và phân tích đặc trưng tiếng nói

Notebook này tạo các hình minh họa đặc trưng từ **audio thật** trong dataset của project cho ba lớp `Northern`, `Central`, và `Southern`. Mục tiêu là hỗ trợ phần báo cáo/thuyết trình: quan sát waveform, MFCC, log-Mel spectrogram, nghe lại audio đã tiền xử lý, và giải thích vì sao các đặc trưng này phù hợp với các mô hình trong project.

Các kết quả được ghi ra:

- `outputs/audio/feature_visualization/`
- `outputs/figures/feature_visualization/`
- `outputs/reports/feature_visualization_summary.md`

Notebook được thiết kế để chạy từ thư mục gốc repository. Nếu dataset không nằm trong `data/`, hãy đặt biến môi trường `DATA_ROOT` trỏ đến thư mục chứa dữ liệu trước khi chạy notebook.

## Ý nghĩa của waveform và tiền xử lý

**Waveform** biểu diễn biên độ tín hiệu âm thanh theo thời gian. Trục hoành thường là thời gian tính bằng giây, còn trục tung là biên độ dao động của sóng âm sau khi đọc từ file audio. Waveform giúp ta kiểm tra nhanh các đoạn im lặng, độ lớn tín hiệu, hiện tượng clipping, hoặc sự khác biệt rất tổng quát về nhịp nói và độ dài câu.

Tiền xử lý là cần thiết vì dữ liệu âm thanh có thể khác nhau về sample rate, số kênh, âm lượng, độ dài, và khoảng lặng đầu/cuối. Pipeline của project chuẩn hóa audio về mono, 16 kHz, cắt khoảng lặng, chuẩn hóa âm lượng, rồi pad/crop về độ dài cố định. Điều này giúp các đặc trưng MFCC và log-Mel được tính trên cùng một chuẩn đầu vào, giảm ảnh hưởng của định dạng file và điều kiện thu âm.

Tuy nhiên, waveform **không trực tiếp chứng minh phương ngữ**. Hai waveform khác nhau có thể do người nói khác nhau, câu nói khác nhau, môi trường thu khác nhau, khoảng cách micro, nhiễu nền, hoặc chính nội dung âm vị trong câu.

## MFCC: khái niệm và công thức

**MFCC** (Mel-Frequency Cepstral Coefficients) là đặc trưng tóm tắt dạng bao phổ ngắn hạn của tín hiệu tiếng nói. Thay vì dùng trực tiếp waveform dài, MFCC chia tín hiệu thành các frame ngắn, chuyển sang miền tần số, gom năng lượng theo thang Mel gần với cảm nhận thính giác của con người, lấy log, rồi dùng DCT để tạo các hệ số compact.

Pipeline khái niệm:

- Audio signal: `x[n]`
- Framing and windowing: `x_t[n] = x[n + tH] · w[n]`
- FFT: `X_t[k] = Σ x_t[n] e^{-j2πkn/N}`
- Power spectrum: `P_t[k] = |X_t[k]|²`
- Mel filterbank energy: `E_t[m] = Σ P_t[k] H_m[k]`
- Log compression: `L_t[m] = log(E_t[m] + ε)`
- DCT: `c_t[q] = Σ L_t[m] cos(πq(m − 0.5)/M)`

Trong baseline truyền thống của project, mỗi mẫu audio được biểu diễn bằng vector 26 chiều:

`z = [ μ(c₁), …, μ(c₁₃), σ(c₁), …, σ(c₁₃) ] ∈ R²⁶`

Trong đó `μ` là trung bình theo thời gian và `σ` là độ lệch chuẩn theo thời gian. Vector 26-D này phù hợp với Logistic Regression và SVM vì nó có kích thước nhỏ, cố định, ít tốn tài nguyên, và tóm tắt được phổ tiếng nói theo một dạng mà các mô hình ML truyền thống có thể xử lý trực tiếp.

Khi đọc heatmap MFCC, trục ngang là thời gian/frame, trục dọc là chỉ số hệ số MFCC, và màu sắc biểu diễn giá trị hệ số. Các pattern khác nhau có thể gợi ý khác biệt về phổ/ngữ âm, nhưng vẫn chỉ là phân tích thăm dò.

## Log-Mel spectrogram và CNN

**Log-Mel spectrogram** giữ lại cấu trúc thời gian - tần số rõ hơn so với vector MFCC đã lấy trung bình/độ lệch chuẩn. Mỗi hàng là một Mel bin, mỗi cột là một frame thời gian, và màu sáng hơn thường biểu thị năng lượng mạnh hơn ở vùng thời gian - tần số đó.

Đặc trưng này phù hợp với CNN vì CNN học tốt các pattern cục bộ: dải năng lượng, chuyển động formant, vùng âm hữu thanh/vô thanh, hoặc cấu trúc phổ thay đổi theo thời gian. Trong project, log-Mel spectrogram dùng 64 Mel bins và được chuẩn hóa theo từng mẫu khi đưa vào CNN.

Điều quan trọng: hình ảnh log-Mel hoặc MFCC chỉ giúp quan sát và đặt giả thuyết. Ta không thể kết luận chắc chắn một khác biệt thị giác là do phương ngữ, vì nó cũng có thể đến từ speaker identity, điều kiện thu, nội dung câu, tốc độ nói, giới tính, hoặc nhiễu nền. Các biểu đồ dưới đây là exploratory analysis, không phải bằng chứng khoa học cuối cùng về khác biệt phương ngữ.

In [ ]:
from __future__ import annotations

import csv
import math
import os
import random
import shutil
import sys
import tempfile
from pathlib import Path
from typing import Any

# Keep matplotlib cache inside a writable temp directory when the home cache is locked down.
os.environ.setdefault(
    "MPLCONFIGDIR",
    str(Path(tempfile.gettempdir()) / "vimd_feature_visualization_matplotlib"),
)
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
from IPython.display import Audio, Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    """Find the repository root so the notebook can run from root or notebooks/."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "PLAN.md").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError(
        "Không tìm thấy repository root. Hãy chạy notebook từ thư mục gốc project."
    )


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.features.logmel import DEFAULT_N_MELS, log_mel_spectrogram
from src.features.mfcc import DEFAULT_HOP_LENGTH, DEFAULT_N_MFCC, mfcc_matrix
from src.utils.audio import TARGET_SAMPLE_RATE, load_audio, preprocess_waveform

CLASSES = ["Northern", "Central", "Southern"]
N_SAMPLES_PER_CLASS = 5
RANDOM_STATE = 42
DPI = 300

FIGURE_ROOT = REPO_ROOT / "outputs" / "figures" / "feature_visualization"
INDIVIDUAL_FIGURE_DIR = FIGURE_ROOT / "individual_samples"
AUDIO_OUTPUT_DIR = REPO_ROOT / "outputs" / "audio" / "feature_visualization"
REPORT_OUTPUT_DIR = REPO_ROOT / "outputs" / "reports"
SUMMARY_REPORT_PATH = REPORT_OUTPUT_DIR / "feature_visualization_summary.md"

for directory in [FIGURE_ROOT, INDIVIDUAL_FIGURE_DIR, AUDIO_OUTPUT_DIR, REPORT_OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "font.family": "serif",
        "font.serif": ["Times New Roman", "DejaVu Serif", "Liberation Serif"],
        "font.size": 11,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
    }
)

print(f"Repository root: {REPO_ROOT}")
print("Output folders:")
for directory in [FIGURE_ROOT, INDIVIDUAL_FIGURE_DIR, AUDIO_OUTPUT_DIR, REPORT_OUTPUT_DIR]:
    print(f"- {directory.relative_to(REPO_ROOT)}")


In [ ]:
AUDIO_EXTENSIONS = {".wav", ".flac", ".mp3", ".ogg", ".m4a", ".aac"}
LABEL_ALIASES = {
    "northern": "Northern",
    "north": "Northern",
    "bac": "Northern",
    "bắc": "Northern",
    "central": "Central",
    "center": "Central",
    "middle": "Central",
    "trung": "Central",
    "southern": "Southern",
    "south": "Southern",
    "nam": "Southern",
}
LABEL_COLUMNS = ["label", "dialect", "dialect_label", "class", "class_label", "region", "source_region"]
SPLIT_COLUMNS = ["source_split", "split", "subset", "partition"]
SAMPLE_ID_COLUMNS = ["sample_id", "id", "utt_id", "utterance_id", "filename", "file_name"]
PREPROCESSED_PATH_COLUMNS = ["preprocessed_audio_path", "processed_audio_path"]
ORIGINAL_PATH_COLUMNS = [
    "audio_path",
    "original_audio_path",
    "path",
    "filepath",
    "file_path",
    "wav_path",
    "source_audio_path",
]
DURATION_COLUMNS = ["original_duration_seconds", "duration_seconds", "duration", "audio_duration"]


def relpath(path: str | Path | None) -> str:
    if path is None:
        return ""
    p = Path(path)
    try:
        return str(p.resolve().relative_to(REPO_ROOT))
    except Exception:
        return str(p)


def normalize_label(value: str | None) -> str | None:
    if not value:
        return None
    text = str(value).strip()
    if text in CLASSES:
        return text
    return LABEL_ALIASES.get(text.lower())


def row_label(row: dict[str, str]) -> str | None:
    for column in LABEL_COLUMNS:
        label = normalize_label(row.get(column))
        if label:
            return label
    return None


def row_split(row: dict[str, str]) -> str:
    for column in SPLIT_COLUMNS:
        value = (row.get(column) or "").strip().lower()
        if value:
            return value
    return "unknown"


def first_present(row: dict[str, str], columns: list[str]) -> str:
    for column in columns:
        value = (row.get(column) or "").strip()
        if value:
            return value
    return ""


def parse_duration(row: dict[str, str]) -> float | None:
    for column in DURATION_COLUMNS:
        value = (row.get(column) or "").strip()
        if not value:
            continue
        try:
            duration = float(value)
        except ValueError:
            continue
        if math.isfinite(duration) and duration > 0:
            return duration
    return None


def data_roots() -> list[Path]:
    roots: list[Path] = []
    env_root = os.environ.get("DATA_ROOT")
    if env_root:
        roots.append(Path(env_root).expanduser())
    roots.extend([REPO_ROOT / "data" / "processed", REPO_ROOT / "data" / "raw", REPO_ROOT / "data"])

    unique: list[Path] = []
    seen: set[Path] = set()
    for root in roots:
        resolved = root.resolve()
        if resolved not in seen and resolved.exists():
            seen.add(resolved)
            unique.append(resolved)
    return unique


def manifest_candidates() -> list[Path]:
    preferred = [
        REPO_ROOT / "data" / "processed" / "preprocessed_metadata.csv",
        REPO_ROOT / "data" / "processed" / "metadata_clean.csv",
        REPO_ROOT / "data" / "metadata.csv",
    ]
    discovered: list[Path] = []
    for root in data_roots():
        discovered.extend(root.rglob("*.csv"))

    def priority(path: Path) -> tuple[int, int, str]:
        name = path.name.lower()
        if name == "preprocessed_metadata.csv":
            rank = 0
        elif name == "metadata_clean.csv":
            rank = 1
        elif "metadata" in name:
            rank = 2
        elif "split" in name:
            rank = 3
        else:
            rank = 4
        return rank, len(path.parts), str(path)

    candidates = [path for path in preferred if path.exists()]
    candidates.extend(discovered)
    unique = []
    seen: set[Path] = set()
    for path in sorted(candidates, key=priority):
        resolved = path.resolve()
        if resolved not in seen:
            seen.add(resolved)
            unique.append(resolved)
    return unique


def resolve_path(value: str, manifest_path: Path | None = None) -> Path | None:
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none", "null"}:
        return None
    candidate = Path(text).expanduser()
    if candidate.is_absolute():
        return candidate if candidate.exists() else None

    bases = [REPO_ROOT]
    if manifest_path is not None:
        bases.append(manifest_path.parent)
    for root in data_roots():
        bases.append(root)

    for base in bases:
        path = (base / candidate).resolve()
        if path.exists():
            return path
    return None


def path_from_columns(
    row: dict[str, str], columns: list[str], manifest_path: Path | None
) -> Path | None:
    for column in columns:
        value = row.get(column)
        if value:
            path = resolve_path(value, manifest_path)
            if path and path.suffix.lower() in AUDIO_EXTENSIONS:
                return path
    return None


def rows_from_manifest(path: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with path.open(encoding="utf-8", newline="") as input_file:
        reader = csv.DictReader(input_file)
        if not reader.fieldnames:
            return rows
        for row in reader:
            label = row_label(row)
            if label not in CLASSES:
                continue
            preprocessed_path = path_from_columns(row, PREPROCESSED_PATH_COLUMNS, path)
            original_path = path_from_columns(row, ORIGINAL_PATH_COLUMNS, path)
            loadable_path = preprocessed_path or original_path
            if loadable_path is None:
                continue
            sample_id = first_present(row, SAMPLE_ID_COLUMNS) or loadable_path.stem
            rows.append(
                {
                    "sample_id": sample_id,
                    "label": label,
                    "split": row_split(row),
                    "manifest_path": path,
                    "original_path": original_path,
                    "preprocessed_path": preprocessed_path,
                    "loadable_path": loadable_path,
                    "duration_from_metadata": parse_duration(row),
                }
            )
    return rows


def rows_from_audio_folders() -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    seen: set[Path] = set()
    for root in data_roots():
        for path in root.rglob("*"):
            if path.suffix.lower() not in AUDIO_EXTENSIONS or not path.is_file():
                continue
            resolved = path.resolve()
            if resolved in seen:
                continue
            seen.add(resolved)
            parts = {part.lower(): part for part in path.parts}
            label = None
            for class_name in CLASSES:
                if class_name.lower() in parts:
                    label = class_name
                    break
            if label is None:
                continue
            split = "unknown"
            for candidate in ["train", "valid", "validation", "test"]:
                if candidate in parts:
                    split = "valid" if candidate == "validation" else candidate
                    break
            rows.append(
                {
                    "sample_id": path.stem,
                    "label": label,
                    "split": split,
                    "manifest_path": None,
                    "original_path": path,
                    "preprocessed_path": None,
                    "loadable_path": path,
                    "duration_from_metadata": None,
                }
            )
    return rows


def load_candidate_rows() -> tuple[list[dict[str, Any]], str]:
    for manifest in manifest_candidates():
        rows = rows_from_manifest(manifest)
        if rows:
            return rows, f"manifest: {relpath(manifest)}"
    rows = rows_from_audio_folders()
    if rows:
        return rows, "audio folder scan"
    roots_text = ", ".join(relpath(root) for root in data_roots()) or "data/"
    raise FileNotFoundError(
        "Không tìm thấy audio thật cho ba lớp Northern/Central/Southern trong các vị trí phổ biến. "
        f"Đã kiểm tra: {roots_text}. Hãy đặt biến môi trường DATA_ROOT trỏ tới thư mục dataset rồi chạy lại notebook."
    )


CANDIDATE_ROWS, DATA_SOURCE_DESCRIPTION = load_candidate_rows()

train_rows = [row for row in CANDIDATE_ROWS if row["split"] == "train"]
if all(any(row["label"] == label for row in train_rows) for label in CLASSES):
    SELECTION_POOL = train_rows
    SELECTION_SPLIT_DESCRIPTION = "train"
else:
    SELECTION_POOL = CANDIDATE_ROWS
    SELECTION_SPLIT_DESCRIPTION = "all available splits"

rng = random.Random(RANDOM_STATE)
SELECTED_ROWS: list[dict[str, Any]] = []
for label in CLASSES:
    class_rows = sorted(
        [row for row in SELECTION_POOL if row["label"] == label],
        key=lambda item: (item["sample_id"], relpath(item["loadable_path"])),
    )
    if not class_rows:
        raise FileNotFoundError(
            f"Không tìm thấy audio cho lớp {label}. Hãy kiểm tra dataset hoặc đặt DATA_ROOT thủ công."
        )
    if len(class_rows) < N_SAMPLES_PER_CLASS:
        print(
            f"Warning: lớp {label} chỉ có {len(class_rows)} mẫu; dùng toàn bộ thay vì {N_SAMPLES_PER_CLASS}."
        )
        chosen = class_rows
    else:
        chosen = rng.sample(class_rows, N_SAMPLES_PER_CLASS)
    for index, row in enumerate(chosen, start=1):
        selected = dict(row)
        selected["class_index"] = index
        SELECTED_ROWS.append(selected)

print(f"Data source: {DATA_SOURCE_DESCRIPTION}")
print(f"Selection split: {SELECTION_SPLIT_DESCRIPTION}")
print(f"Random state: {RANDOM_STATE}")
for label in CLASSES:
    count = sum(1 for row in SELECTED_ROWS if row["label"] == label)
    print(f"Selected {count} sample(s) for {label}")


In [ ]:
def validate_saved_file(path: Path, suffix: str | None = None) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Expected file was not saved: {path}")
    if path.stat().st_size <= 0:
        raise ValueError(f"Saved file is empty: {path}")
    if suffix and path.suffix.lower() != suffix.lower():
        raise ValueError(f"Expected {suffix} file, received: {path}")


def export_wav(path: Path, waveform: np.ndarray, sample_rate: int) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(path, waveform.astype(np.float32), sample_rate, format="WAV", subtype="PCM_16")
    validate_saved_file(path, ".wav")
    return path


def safe_filename_label(label: str) -> str:
    return "".join(ch if ch.isalnum() else "_" for ch in label).strip("_")


SAMPLES: list[dict[str, Any]] = []
for row in SELECTED_ROWS:
    label = row["label"]
    index = int(row["class_index"])
    filename_label = safe_filename_label(label)
    original_path = row.get("original_path") or row.get("loadable_path")
    if original_path is None:
        raise FileNotFoundError(f"Không có audio source cho mẫu {row['sample_id']}")

    original_waveform, original_sr = load_audio(original_path)
    preprocessed_waveform, stats = preprocess_waveform(original_waveform, original_sr)

    preprocessed_export_path = AUDIO_OUTPUT_DIR / f"{filename_label}_sample_{index:02d}_preprocessed.wav"
    original_export_path = AUDIO_OUTPUT_DIR / f"{filename_label}_sample_{index:02d}_original.wav"
    export_wav(preprocessed_export_path, preprocessed_waveform, TARGET_SAMPLE_RATE)
    export_wav(original_export_path, original_waveform, original_sr)

    mfcc = mfcc_matrix(
        preprocessed_waveform,
        sample_rate=TARGET_SAMPLE_RATE,
        n_mfcc=DEFAULT_N_MFCC,
    )
    mfcc_feature_vector = np.concatenate([mfcc.mean(axis=0), mfcc.std(axis=0)]).astype(np.float32)
    logmel_for_cnn = log_mel_spectrogram(
        preprocessed_waveform,
        sample_rate=TARGET_SAMPLE_RATE,
        n_mels=DEFAULT_N_MELS,
        standardize=True,
    )
    logmel_energy = log_mel_spectrogram(
        preprocessed_waveform,
        sample_rate=TARGET_SAMPLE_RATE,
        n_mels=DEFAULT_N_MELS,
        standardize=False,
    ).mean(axis=1)

    SAMPLES.append(
        {
            "sample_id": row["sample_id"],
            "label": label,
            "class_index": index,
            "original_path": original_path,
            "source_preprocessed_path": row.get("preprocessed_path"),
            "original_export_path": original_export_path,
            "preprocessed_export_path": preprocessed_export_path,
            "original_duration_seconds": len(original_waveform) / float(original_sr),
            "preprocessed_duration_seconds": stats.output_duration_seconds,
            "sample_rate": int(original_sr),
            "preprocessed_sample_rate": TARGET_SAMPLE_RATE,
            "waveform": preprocessed_waveform,
            "mfcc": mfcc,
            "mfcc_feature_vector": mfcc_feature_vector,
            "logmel_for_cnn": logmel_for_cnn,
            "logmel_energy": logmel_energy,
        }
    )

print(f"Exported {len(SAMPLES) * 2} audio files to {relpath(AUDIO_OUTPUT_DIR)}")
print("MFCC shape example:", SAMPLES[0]["mfcc"].shape, "(time frames, coefficients)")
print("Log-Mel shape example:", SAMPLES[0]["logmel_for_cnn"].shape, "(Mel bins, time frames)")


## Bảng mẫu được chọn và audio để nghe so sánh

Bảng dưới đây ghi lại sample ID, lớp phương ngữ, đường dẫn audio gốc, thời lượng gốc, thời lượng sau tiền xử lý, sample rate, và file audio đã export. Mỗi mẫu có audio player cho bản đã tiền xử lý và bản gốc được export lại, giúp nghe đối chiếu với waveform, MFCC và log-Mel spectrogram.

Khi nghe các file này, nên chú ý rằng mục tiêu là so sánh dữ liệu đầu vào và đặc trưng, không phải suy luận danh tính người nói hay quê quán cá nhân.

In [ ]:
def markdown_table(rows: list[dict[str, Any]], columns: list[tuple[str, str]]) -> str:
    headers = [header for header, _ in columns]
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        values = []
        for _, key in columns:
            value = row.get(key, "")
            if isinstance(value, float):
                value = f"{value:.2f}"
            value = str(value).replace("\n", " ").replace("|", "\\|")
            values.append(value)
        lines.append("| " + " | ".join(values) + " |")
    return "\n".join(lines)


SAMPLE_TABLE_ROWS = [
    {
        "sample_id": sample["sample_id"],
        "dialect": sample["label"],
        "original_path": relpath(sample["original_path"]),
        "original_duration_seconds": sample["original_duration_seconds"],
        "preprocessed_duration_seconds": sample["preprocessed_duration_seconds"],
        "sample_rate": sample["sample_rate"],
        "exported_audio_path": relpath(sample["preprocessed_export_path"]),
    }
    for sample in SAMPLES
]

display(
    Markdown(
        markdown_table(
            SAMPLE_TABLE_ROWS,
            [
                ("sample ID", "sample_id"),
                ("dialect", "dialect"),
                ("original file path", "original_path"),
                ("original duration", "original_duration_seconds"),
                ("preprocessed duration", "preprocessed_duration_seconds"),
                ("sample rate", "sample_rate"),
                ("exported audio path", "exported_audio_path"),
            ],
        )
    )
)

for sample in SAMPLES:
    display(
        Markdown(
            f"### {sample['label']} — sample {sample['class_index']:02d}\n"
            f"- Sample ID: `{sample['sample_id']}`\n"
            f"- Original path: `{relpath(sample['original_path'])}`\n"
            f"- Preprocessed export: `{relpath(sample['preprocessed_export_path'])}`\n"
        )
    )
    display(Audio(url=f"/files/{relpath(sample['preprocessed_export_path'])}", embed=False))
    display(Markdown(f"Original/exported source: `{relpath(sample['original_export_path'])}`"))
    display(Audio(url=f"/files/{relpath(sample['original_export_path'])}", embed=False))


In [ ]:
CLASS_COLORS = {
    "Northern": "#1f77b4",
    "Central": "#2ca02c",
    "Southern": "#d62728",
}
GENERATED_FIGURES: list[dict[str, str]] = []
OPTIONAL_NOTES: list[str] = []


def plot_waveform(ax: plt.Axes, waveform: np.ndarray, sample_rate: int, title: str = "") -> None:
    max_points = 5000
    step = max(1, int(np.ceil(len(waveform) / max_points)))
    time_axis = np.arange(len(waveform))[::step] / float(sample_rate)
    ax.plot(time_axis, waveform[::step], linewidth=0.6, color="#2b2b2b")
    ax.set_title(title)
    ax.set_xlabel("Time (seconds)")
    ax.set_ylabel("Amplitude")
    ax.set_ylim(-1.05, 1.05)
    ax.grid(True, linewidth=0.3, alpha=0.35)


def feature_extent(feature_time_frames: int, sample_rate: int = TARGET_SAMPLE_RATE) -> list[float]:
    duration = max(0.0, (feature_time_frames - 1) * DEFAULT_HOP_LENGTH / float(sample_rate))
    return [0.0, duration, 0.5, 0.5]


def save_current_figure(fig: plt.Figure, path: Path, purpose: str, kind: str) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=DPI, bbox_inches="tight")
    plt.close(fig)
    validate_saved_file(path, ".png")
    GENERATED_FIGURES.append(
        {"figure_name": path.name, "file_path": relpath(path), "purpose": purpose, "kind": kind}
    )
    return path


for sample in SAMPLES:
    label = sample["label"]
    index = sample["class_index"]
    waveform = sample["waveform"]
    mfcc = sample["mfcc"]
    logmel = sample["logmel_for_cnn"]
    duration = len(waveform) / float(TARGET_SAMPLE_RATE)

    fig, axes = plt.subplots(1, 3, figsize=(18, 4.8), constrained_layout=True)
    plot_waveform(axes[0], waveform, TARGET_SAMPLE_RATE, "Waveform after preprocessing")

    mfcc_image = axes[1].imshow(
        mfcc.T,
        aspect="auto",
        origin="lower",
        interpolation="nearest",
        cmap="viridis",
        extent=[0, duration, 1, mfcc.shape[1]],
    )
    axes[1].set_title("MFCC heatmap (13 coefficients)")
    axes[1].set_xlabel("Time (seconds)")
    axes[1].set_ylabel("MFCC coefficient index")
    fig.colorbar(mfcc_image, ax=axes[1], fraction=0.046, pad=0.04)

    logmel_image = axes[2].imshow(
        logmel,
        aspect="auto",
        origin="lower",
        interpolation="nearest",
        cmap="magma",
        extent=[0, duration, 0, logmel.shape[0]],
    )
    axes[2].set_title("Log-Mel spectrogram (64 Mel bins)")
    axes[2].set_xlabel("Time (seconds)")
    axes[2].set_ylabel("Mel bin index")
    fig.colorbar(logmel_image, ax=axes[2], fraction=0.046, pad=0.04)

    fig.suptitle(f"{label} sample {index:02d}: waveform, MFCC, and log-Mel", fontsize=15)
    output_path = INDIVIDUAL_FIGURE_DIR / f"{safe_filename_label(label)}_sample_{index:02d}_features.png"
    save_current_figure(
        fig,
        output_path,
        f"Waveform, MFCC, and log-Mel panels for {label} sample {index:02d}.",
        "individual",
    )

print(f"Generated {sum(1 for item in GENERATED_FIGURES if item['kind'] == 'individual')} individual sample figures.")


## Diễn giải các hình từng mẫu

Mỗi hình từng mẫu gồm ba panel:

- **Waveform** cho thấy biên độ theo thời gian sau tiền xử lý. Hình này hữu ích để kiểm tra khoảng lặng, độ lớn tín hiệu, và hình dạng tổng quát của tín hiệu, nhưng không tự nó chỉ ra phương ngữ.
- **MFCC heatmap** tóm tắt bao phổ ngắn hạn. Các pattern màu có thể phản ánh phát âm, nội dung âm vị, người nói, hoặc môi trường thu. MFCC compact nên phù hợp với Logistic Regression và SVM trong baseline.
- **Log-Mel spectrogram** giữ cấu trúc thời gian - tần số nhiều hơn, giúp CNN học các pattern cục bộ theo thời gian và theo dải tần. Vùng sáng hơn thường tương ứng với năng lượng tương đối mạnh hơn trong biểu diễn đang xem.

Các hình này giúp minh họa đặc trưng đã trích xuất, nhưng chỉ nên dùng như evidence trực quan cho phân tích thăm dò.

In [ ]:
SAMPLES_BY_CLASS = {label: [sample for sample in SAMPLES if sample["label"] == label] for label in CLASSES}
MAX_COLUMNS = max(len(samples) for samples in SAMPLES_BY_CLASS.values())


def make_axes_grid(figsize: tuple[float, float]) -> tuple[plt.Figure, np.ndarray]:
    fig, axes = plt.subplots(len(CLASSES), MAX_COLUMNS, figsize=figsize, squeeze=False, constrained_layout=True)
    return fig, axes


# Figure 1: waveform grid
fig, axes = make_axes_grid((3.4 * MAX_COLUMNS, 8.5))
for row_index, label in enumerate(CLASSES):
    for col_index in range(MAX_COLUMNS):
        ax = axes[row_index, col_index]
        class_samples = SAMPLES_BY_CLASS[label]
        if col_index >= len(class_samples):
            ax.axis("off")
            continue
        sample = class_samples[col_index]
        plot_waveform(ax, sample["waveform"], TARGET_SAMPLE_RATE, f"sample {sample['class_index']:02d}")
        if col_index == 0:
            ax.set_ylabel(f"{label}\nAmplitude")
fig.suptitle("Waveform examples after preprocessing", fontsize=16)
save_current_figure(
    fig,
    FIGURE_ROOT / "01_waveform_grid.png",
    "Compare preprocessed waveform shapes across selected samples and dialect classes.",
    "summary",
)

# Figure 2: MFCC heatmap grid
fig, axes = make_axes_grid((3.4 * MAX_COLUMNS, 8.5))
for row_index, label in enumerate(CLASSES):
    for col_index in range(MAX_COLUMNS):
        ax = axes[row_index, col_index]
        class_samples = SAMPLES_BY_CLASS[label]
        if col_index >= len(class_samples):
            ax.axis("off")
            continue
        sample = class_samples[col_index]
        duration = len(sample["waveform"]) / float(TARGET_SAMPLE_RATE)
        image = ax.imshow(
            sample["mfcc"].T,
            aspect="auto",
            origin="lower",
            interpolation="nearest",
            cmap="viridis",
            extent=[0, duration, 1, sample["mfcc"].shape[1]],
        )
        ax.set_title(f"sample {sample['class_index']:02d}")
        ax.set_xlabel("Time (s)")
        if col_index == 0:
            ax.set_ylabel(f"{label}\nMFCC index")
fig.suptitle("MFCC feature maps", fontsize=16)
save_current_figure(
    fig,
    FIGURE_ROOT / "02_mfcc_heatmap_grid.png",
    "Compare 13-coefficient MFCC maps across selected dialect samples.",
    "summary",
)

# Figure 3: log-Mel spectrogram grid
fig, axes = make_axes_grid((3.4 * MAX_COLUMNS, 8.5))
for row_index, label in enumerate(CLASSES):
    for col_index in range(MAX_COLUMNS):
        ax = axes[row_index, col_index]
        class_samples = SAMPLES_BY_CLASS[label]
        if col_index >= len(class_samples):
            ax.axis("off")
            continue
        sample = class_samples[col_index]
        duration = len(sample["waveform"]) / float(TARGET_SAMPLE_RATE)
        image = ax.imshow(
            sample["logmel_for_cnn"],
            aspect="auto",
            origin="lower",
            interpolation="nearest",
            cmap="magma",
            extent=[0, duration, 0, sample["logmel_for_cnn"].shape[0]],
        )
        ax.set_title(f"sample {sample['class_index']:02d}")
        ax.set_xlabel("Time (s)")
        if col_index == 0:
            ax.set_ylabel(f"{label}\nMel bin")
fig.suptitle("Log-Mel spectrogram examples", fontsize=16)
save_current_figure(
    fig,
    FIGURE_ROOT / "03_logmel_spectrogram_grid.png",
    "Compare 64-bin standardized log-Mel spectrograms used by the CNN pipeline.",
    "summary",
)

print("Generated waveform, MFCC, and log-Mel class-level grids.")


## Diễn giải các hình so sánh theo lớp

Ở waveform grid, mỗi hàng là một phương ngữ và mỗi cột là một mẫu đã chọn. Hình này cho phép kiểm tra độ lớn tín hiệu, khoảng lặng, và nhịp thay đổi biên độ sau tiền xử lý. Nó không đủ để kết luận phương ngữ vì waveform phụ thuộc mạnh vào câu nói, người nói và điều kiện thu.

Ở MFCC grid, các heatmap cho thấy cách bao phổ thay đổi theo thời gian. Nếu thấy một vài vùng màu khác nhau giữa các hàng, ta chỉ nên xem đó là gợi ý ban đầu. Khác biệt có thể đến từ dialect, nhưng cũng có thể đến từ speaker identity, nội dung câu, giới tính, tốc độ nói, hoặc nhiễu.

Ở log-Mel grid, cấu trúc thời gian - tần số được giữ lại nhiều hơn. Đây là lý do biểu diễn này phù hợp với CNN: CNN có thể học các pattern cục bộ trong từng vùng thời gian và Mel bin. Các hình vẫn là phân tích khám phá, không phải bằng chứng cuối cùng về khác biệt phương ngữ.

In [ ]:
# Figure 4: MFCC 26-D mean + std vectors by sample
fig, axes = plt.subplots(len(CLASSES), 1, figsize=(13, 8.5), sharex=True, constrained_layout=True)
dimensions = np.arange(1, 2 * DEFAULT_N_MFCC + 1)
for ax, label in zip(axes, CLASSES):
    for sample in SAMPLES_BY_CLASS[label]:
        vector = sample["mfcc_feature_vector"]
        ax.plot(
            dimensions,
            vector,
            linewidth=1.4,
            marker="o",
            markersize=2.5,
            alpha=0.85,
            label=f"sample {sample['class_index']:02d}",
        )
    ax.axvline(DEFAULT_N_MFCC + 0.5, color="#555555", linestyle="--", linewidth=1)
    ax.text(3.0, ax.get_ylim()[1] * 0.85, "MFCC mean", fontsize=10)
    ax.text(DEFAULT_N_MFCC + 3.0, ax.get_ylim()[1] * 0.85, "MFCC std", fontsize=10)
    ax.set_ylabel(label)
    ax.grid(True, linewidth=0.3, alpha=0.35)
    ax.legend(loc="best", ncols=min(5, len(SAMPLES_BY_CLASS[label])))
axes[-1].set_xlabel("Feature dimension")
fig.suptitle("MFCC mean + standard deviation feature vectors by sample", fontsize=16)
save_current_figure(
    fig,
    FIGURE_ROOT / "04_mfcc_mean_std_vectors_by_sample.png",
    "Show each selected sample as the 26-D MFCC mean plus standard deviation vector used by traditional ML models.",
    "summary",
)

# Figure 5: average MFCC vector by class
fig, ax = plt.subplots(figsize=(13, 5.5), constrained_layout=True)
for label in CLASSES:
    vectors = np.stack([sample["mfcc_feature_vector"] for sample in SAMPLES_BY_CLASS[label]])
    ax.plot(
        dimensions,
        vectors.mean(axis=0),
        marker="o",
        linewidth=2.0,
        markersize=3.5,
        color=CLASS_COLORS[label],
        label=label,
    )
ax.axvline(DEFAULT_N_MFCC + 0.5, color="#555555", linestyle="--", linewidth=1)
ax.text(3.0, ax.get_ylim()[1] * 0.85, "MFCC mean", fontsize=10)
ax.text(DEFAULT_N_MFCC + 3.0, ax.get_ylim()[1] * 0.85, "MFCC std", fontsize=10)
ax.set_xlabel("Feature dimension")
ax.set_ylabel("Average feature value")
ax.grid(True, linewidth=0.3, alpha=0.35)
ax.legend(loc="best")
fig.suptitle("Average MFCC feature vector by dialect class", fontsize=16)
save_current_figure(
    fig,
    FIGURE_ROOT / "05_mfcc_class_average_vector.png",
    "Compare class-level averages of selected 26-D MFCC feature vectors.",
    "summary",
)

print("Generated MFCC feature vector comparison figures.")


## Diễn giải vector MFCC

Vector 26 chiều trong hai hình trên chính là biểu diễn compact dùng cho Logistic Regression và SVM: 13 chiều đầu là trung bình của từng hệ số MFCC theo thời gian, 13 chiều sau là độ lệch chuẩn theo thời gian. Vì mọi mẫu đều có cùng số chiều, các mô hình truyền thống có thể học decision boundary trực tiếp trên vector này.

Hình theo từng mẫu cho thấy mức biến thiên giữa các người nói và câu nói trong cùng một lớp. Hình trung bình theo lớp giúp so sánh xu hướng rộng giữa các mẫu đã chọn. Tuy nhiên, vì chỉ dùng tối đa 5 mẫu mỗi lớp, không nên diễn giải các đường trung bình này như kết luận thống kê chắc chắn.

In [ ]:
# Figure 6: average log-Mel energy by class
fig, ax = plt.subplots(figsize=(12, 5.5), constrained_layout=True)
mel_bins = np.arange(DEFAULT_N_MELS)
for label in CLASSES:
    energy_vectors = np.stack([sample["logmel_energy"] for sample in SAMPLES_BY_CLASS[label]])
    ax.plot(
        mel_bins,
        energy_vectors.mean(axis=0),
        linewidth=2.0,
        color=CLASS_COLORS[label],
        label=label,
    )
ax.set_xlabel("Mel bin index")
ax.set_ylabel("Average log-Mel energy")
ax.grid(True, linewidth=0.3, alpha=0.35)
ax.legend(loc="best")
fig.suptitle("Average log-Mel energy profile by dialect class", fontsize=16)
save_current_figure(
    fig,
    FIGURE_ROOT / "06_average_logmel_energy_by_class.png",
    "Compare average raw log-Mel energy profiles across selected samples for each dialect class.",
    "summary",
)

# Figure 7: optional duration distribution by class
DURATION_BY_CLASS = {label: [] for label in CLASSES}
for row in SELECTION_POOL:
    duration = row.get("duration_from_metadata")
    if duration is not None and row["label"] in DURATION_BY_CLASS:
        DURATION_BY_CLASS[row["label"]].append(duration)

if all(DURATION_BY_CLASS[label] for label in CLASSES):
    fig, ax = plt.subplots(figsize=(10, 5.5), constrained_layout=True)
    values = [DURATION_BY_CLASS[label] for label in CLASSES]
    box = ax.boxplot(values, tick_labels=CLASSES, patch_artist=True, showfliers=False)
    for patch, label in zip(box["boxes"], CLASSES):
        patch.set_facecolor(CLASS_COLORS[label])
        patch.set_alpha(0.25)
    ax.set_xlabel("Dialect class")
    ax.set_ylabel("Original duration (seconds)")
    ax.grid(True, axis="y", linewidth=0.3, alpha=0.35)
    fig.suptitle("Original audio duration distribution by dialect class", fontsize=16)
    save_current_figure(
        fig,
        FIGURE_ROOT / "07_duration_distribution_by_class.png",
        "Show original duration distribution by class using metadata from the selected split.",
        "summary",
    )
else:
    note = "Không tạo được 07_duration_distribution_by_class.png vì metadata không có duration cho đủ ba lớp."
    OPTIONAL_NOTES.append(note)
    print("Optional figure skipped:", note)

print("Generated log-Mel energy figure and optional duration analysis when metadata allowed it.")


## Diễn giải log-Mel energy và duration

Hình log-Mel energy trung bình theo lớp nén mỗi spectrogram theo trục thời gian, còn lại một profile năng lượng theo Mel bin. Profile này giúp quan sát xu hướng phổ rộng của các mẫu đã chọn. Vì audio đã được chuẩn hóa âm lượng, sự khác biệt lớn vẫn cần được đọc cẩn thận: nó có thể đến từ nội dung câu, giọng người nói, hoặc điều kiện thu.

Hình duration distribution, nếu được tạo, cho biết phân bố thời lượng gốc của audio trong split được dùng để chọn mẫu. Duration giúp kiểm tra dữ liệu và bias tiềm ẩn: nếu một lớp có thời lượng rất khác lớp khác, mô hình có thể học các tín hiệu ngoài phương ngữ.

In [ ]:
def generated_audio_rows() -> list[dict[str, str]]:
    rows: list[dict[str, str]] = []
    for sample in SAMPLES:
        rows.append(
            {
                "dialect": sample["label"],
                "sample_id": sample["sample_id"],
                "audio_path": relpath(sample["preprocessed_export_path"]),
                "purpose": "Bản audio đã tiền xử lý để nghe cùng waveform/MFCC/log-Mel.",
            }
        )
        rows.append(
            {
                "dialect": sample["label"],
                "sample_id": sample["sample_id"],
                "audio_path": relpath(sample["original_export_path"]),
                "purpose": "Bản audio gốc được export lại để so sánh trước/sau tiền xử lý.",
            }
        )
    return rows


FIGURE_TABLE_ROWS = [
    {
        "figure_name": item["figure_name"],
        "file_path": item["file_path"],
        "purpose": item["purpose"],
    }
    for item in GENERATED_FIGURES
]
AUDIO_TABLE_ROWS = generated_audio_rows()

report_parts = [
    "# Feature Visualization Summary",
    "",
    "## Purpose",
    "",
    "Báo cáo này tổng hợp các audio và hình ảnh đặc trưng được tạo từ mẫu thật trong dataset của project. Mục tiêu là hỗ trợ mục 2.5 của báo cáo: minh họa và phân tích waveform, MFCC, và log-Mel spectrogram bằng ví dụ, hình ảnh, và biểu đồ có thể dùng trong slide thuyết trình.",
    "",
    "## Dataset/sample selection",
    "",
    f"Notebook đọc dữ liệu từ `{DATA_SOURCE_DESCRIPTION}` và ưu tiên split `{SELECTION_SPLIT_DESCRIPTION}`. Mỗi lớp được chọn tối đa `{N_SAMPLES_PER_CLASS}` mẫu với `random_state = {RANDOM_STATE}`. Nếu một lớp có ít hơn số mẫu yêu cầu, notebook dùng toàn bộ mẫu hiện có và in cảnh báo.",
    "",
    markdown_table(
        SAMPLE_TABLE_ROWS,
        [
            ("sample ID", "sample_id"),
            ("dialect", "dialect"),
            ("original path", "original_path"),
            ("exported audio path", "exported_audio_path"),
            ("duration", "original_duration_seconds"),
            ("sample rate", "sample_rate"),
        ],
    ),
    "",
    "## Generated audio files",
    "",
    "Các file audio đã tiền xử lý dùng để nghe trực tiếp khi so sánh với đặc trưng; các file original giúp kiểm tra khác biệt trước và sau tiền xử lý.",
    "",
    markdown_table(
        AUDIO_TABLE_ROWS,
        [
            ("dialect", "dialect"),
            ("sample ID", "sample_id"),
            ("audio path", "audio_path"),
            ("purpose", "purpose"),
        ],
    ),
    "",
    "## Generated figures",
    "",
    markdown_table(
        FIGURE_TABLE_ROWS,
        [
            ("figure name", "figure_name"),
            ("file path", "file_path"),
            ("purpose", "purpose"),
        ],
    ),
    "",
    "## Waveform analysis",
    "",
    "Waveform biểu diễn biên độ tín hiệu theo thời gian. Nó giúp kiểm tra khoảng lặng, độ lớn tương đối, clipping, và hình dạng tổng quát sau tiền xử lý. Waveform không trực tiếp tiết lộ phương ngữ; các khác biệt quan sát được có thể do câu nói, người nói, micro, nhiễu nền hoặc âm lượng ban đầu.",
    "",
    "## MFCC analysis",
    "",
    "MFCC tóm tắt bao phổ ngắn hạn của tiếng nói. Project dùng 13 hệ số MFCC, sau đó lấy trung bình và độ lệch chuẩn theo thời gian để tạo vector 26 chiều cho Logistic Regression và SVM. Cách biểu diễn này nhỏ gọn, cố định chiều, và phù hợp với các mô hình truyền thống. Heatmap MFCC giúp quan sát pattern phổ theo thời gian nhưng không đủ để kết luận chắc chắn về phương ngữ.",
    "",
    "## Log-Mel spectrogram analysis",
    "",
    "Log-Mel spectrogram giữ cấu trúc thời gian - tần số với 64 Mel bins. Đây là biểu diễn phù hợp cho CNN vì CNN có thể học các pattern cục bộ theo cả trục thời gian và trục tần số. Trong hình, vùng sáng hơn thường biểu thị năng lượng tương đối mạnh hơn, nhưng cách đọc vẫn phải thận trọng vì nội dung câu và điều kiện thu có ảnh hưởng lớn.",
    "",
    "## Class-level comparison",
    "",
    "Các hình class-level cho phép so sánh nhiều mẫu Northern, Central và Southern trong cùng một bố cục. Các đường trung bình MFCC và log-Mel energy giúp nhìn xu hướng chung của nhóm mẫu được chọn. Vì số mẫu chỉ là một phần nhỏ của dataset, các xu hướng này nên được xem là gợi ý khám phá, không phải kết luận thống kê cuối cùng.",
    "",
    "## Limitations",
    "",
    "- Các biểu đồ chỉ dùng mẫu được chọn để minh họa, không phải toàn bộ dataset.",
    "- Khác biệt hình ảnh có thể đến từ phương ngữ, speaker identity, giới tính, nội dung câu, tốc độ nói, điều kiện thu âm hoặc nhiễu.",
    "- Waveform, MFCC và log-Mel là công cụ phân tích thăm dò; kết luận mô hình cần dựa trên đánh giá định lượng như accuracy, macro F1 và confusion matrix.",
    "- Không dùng các hình này để suy luận danh tính, quê quán cụ thể, hoặc thông tin cá nhân của người nói.",
    "",
    "## How to use these figures in the presentation",
    "",
    "Có thể dùng waveform grid để giới thiệu dữ liệu audio sau tiền xử lý, MFCC grid để giải thích baseline Logistic Regression/SVM, log-Mel grid để giải thích input của CNN, và các hình vector trung bình để minh họa cách đặc trưng được nén thành dạng mô hình có thể học. Khi trình bày, nên nhấn mạnh rằng đây là minh họa trực quan hỗ trợ hiểu pipeline, không phải bằng chứng cuối cùng về khác biệt phương ngữ.",
    "",
    "## Gợi ý lời trình bày",
    "",
    "Trong phần này, em minh họa các đặc trưng được trích xuất từ audio thật của ba vùng Northern, Central và Southern. Trước hết, waveform cho thấy biên độ tín hiệu theo thời gian, giúp kiểm tra khoảng lặng, độ dài và độ lớn của âm thanh sau tiền xử lý. Sau đó, MFCC chuyển tín hiệu sang một biểu diễn compact của bao phổ ngắn hạn; vì mỗi audio được nén thành vector 26 chiều gồm trung bình và độ lệch chuẩn của 13 hệ số, đặc trưng này phù hợp với Logistic Regression và SVM. Với CNN, em dùng log-Mel spectrogram vì biểu diễn này giữ lại cấu trúc thời gian - tần số, cho phép CNN học các pattern cục bộ trong tiếng nói. Tuy nhiên, các hình này chỉ dùng để khám phá và giải thích pipeline. Khác biệt thị giác có thể đến từ phương ngữ, người nói, nội dung câu, điều kiện thu hoặc nhiễu, nên kết luận cuối cùng vẫn cần dựa trên kết quả đánh giá định lượng của mô hình.",
]

if OPTIONAL_NOTES:
    report_parts.extend(["", "## Optional figure notes", ""])
    report_parts.extend([f"- {note}" for note in OPTIONAL_NOTES])

SUMMARY_REPORT_PATH.write_text("\n".join(report_parts) + "\n", encoding="utf-8")
validate_saved_file(SUMMARY_REPORT_PATH, ".md")

print(f"Wrote report: {relpath(SUMMARY_REPORT_PATH)}")


In [ ]:
summary_figures = [item for item in GENERATED_FIGURES if item["kind"] == "summary"]
individual_figures = [item for item in GENERATED_FIGURES if item["kind"] == "individual"]
exported_audio_paths = [sample["preprocessed_export_path"] for sample in SAMPLES] + [
    sample["original_export_path"] for sample in SAMPLES
]

for path in exported_audio_paths:
    validate_saved_file(Path(path), ".wav")
for item in GENERATED_FIGURES:
    validate_saved_file(REPO_ROOT / item["file_path"], ".png")
validate_saved_file(SUMMARY_REPORT_PATH, ".md")

print("Final checklist")
for label in CLASSES:
    print(f"- Number of selected samples for {label}: {len(SAMPLES_BY_CLASS[label])}")
print(f"- Number of exported audio files: {len(exported_audio_paths)}")
print(f"- Number of generated individual feature figures: {len(individual_figures)}")
print(f"- Number of generated summary figures: {len(summary_figures)}")
print(f"- Path to final Markdown report: {relpath(SUMMARY_REPORT_PATH)}")

print("\nGenerated output paths")
print(f"- Audio folder: {relpath(AUDIO_OUTPUT_DIR)}")
print(f"- Figure folder: {relpath(FIGURE_ROOT)}")
print(f"- Individual sample figure folder: {relpath(INDIVIDUAL_FIGURE_DIR)}")
print(f"- Markdown report: {relpath(SUMMARY_REPORT_PATH)}")
for item in GENERATED_FIGURES:
    print(f"- {item['file_path']}")
